# Train the gauge keypoint model on Kaggle

1. Add the dataset: **Add Input → Your Datasets → synth-2d** (the zip of `data/synthetic/synth_2d`, see docs/TRAINING.md).
2. Settings: **Accelerator GPU T4 x2**, **Internet on** (needs a phone-verified Kaggle account).
3. **Save Version → Save & Run All (Commit)** runs it in the background (up to 12 h). When it finishes, download `runs/pose/<NAME>/weights/best.pt` and `runs/pose/<NAME>/results.csv` from the Output tab.

`scripts/train_pose.py` adds three keypoints on the scale arc by default (Version 3, see docs/TRAINING.md); add `--no-arc` to the training command for the 4-keypoint model of Versions 1 and 2.

In [ ]:
EPOCHS = 50
MODEL = "yolo11s-pose.pt"
BATCH = 32          # total over both GPUs
DEVICE = "0,1"      # "0" on a single P100/T4
DEGREES = 30        # random rotation, +/- degrees (Version 1: 0, Version 2: 180)
NAME = "arc"        # run name: Version 1 "base", Version 2 "rot", Version 3 "arc"
FRACTION = 1.0      # e.g. 0.05 with EPOCHS = 1 for a quick check

In [ ]:
!pip install -q "ultralytics>=8.3"
!git clone -q https://github.com/uhyon/mv2npp-gauge-reading.git
%cd mv2npp-gauge-reading
!git log --oneline -1

In [ ]:
import glob, os
# the generated set has labels.jsonl next to data.yaml (a run's data.yaml copy does not)
found = [p for p in glob.glob("/kaggle/input/**/data.yaml", recursive=True)
         if os.path.exists(os.path.join(os.path.dirname(p), "labels.jsonl"))]
assert found, "dataset not attached (Add Input -> synth-2d) or its labels.jsonl is missing"
DATA = os.path.dirname(found[0])
print(DATA, len(os.listdir(f"{DATA}/images/train")), "train images")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python scripts/train_pose.py --data {DATA} --model {MODEL} --epochs {EPOCHS} --batch {BATCH} \
    --device {DEVICE} --workers 4 --fraction {FRACTION} --degrees {DEGREES} --project /kaggle/working/runs/pose --name {NAME}

In [ ]:
import pandas as pd
r = pd.read_csv(f"/kaggle/working/runs/pose/{NAME}/results.csv")
r.columns = r.columns.str.strip()
r[[c for c in r.columns if c.startswith(("epoch", "metrics/"))]].tail(10)

If the session stops early, attach this notebook's own output as an input, run the cells above except the training cell, and resume (the 7-keypoint copy is built again; the keypoint count comes from the checkpoint):
`!python scripts/train_pose.py --resume /kaggle/input/<this notebook output>/runs/pose/{NAME}/weights/last.pt --data {DATA} --project /kaggle/working/runs/pose --name {NAME}`